# Collections



## Part 1 — Shapes

## The landscape in one breath

Scala ships two parallel collection hierarchies — one immutable, one mutable. They live in two different packages: `scala.collection.immutable` and `scala.collection.mutable`. The default import that comes with every Scala program points at the immutable side. So when you write `List`, `Vector`, `Set`, or `Map` in code without any extra import, you are getting the immutable version automatically. The mutable versions require an explicit import.

This is a deliberate language design choice. **Immutable is the easy path. Mutable is opt-in.**

There is also one collection that lives outside both hierarchies — `Array`. `Array` is a thin wrapper over the java virtual machine's primitive array type. It is mutable. We will come back to it near the end of part 1.

## Immutable does not mean slow

A common worry: "if I cannot mutate, every change copies the whole collection — that is wasteful."

The immutable collections share structure under the hood. When you prepend an element to a `List`, the language does **not** copy the entire list — it creates a new cell that points at the existing tail. Same idea applies to `Vector` (which shares branches of its internal tree) and to `Map` (which shares unchanged branches of its hash trie). Producing a new collection that differs slightly from an old one is usually a small operation.

Three reasons to prefer immutable by default:

- **Safe sharing.** An immutable collection can be passed across threads, across functions, across Spark executors, with zero coordination. No locks. No defensive copies.
- **Easier to reason about.** A value that cannot change is one less moving part. The function you read on page twelve produces the same answer whether the function on page fourteen ran first or not.
- **Structural sharing makes derivation cheap.** Spark's entire programming model is built on this assumption — resilient distributed datasets and dataframes are immutable. Every Spark operation produces a new one. Notebook 09 of `apache-spark/` revisits this.

## `List` — the singly-linked default

`List` is Scala's immutable singly-linked list. It is built out of two shapes — the empty list, written `Nil`, and the cons cell, written with the double-colon `::` operator, which prepends one element to another list. So the list `List(1, 2, 3)` is internally `1 :: 2 :: 3 :: Nil`. Each cell holds a value and a pointer to the rest.

```text
   List(1, 2, 3) internally:

   +---+---+    +---+---+    +---+---+    +-----+
   | 1 | *-+--> | 2 | *-+--> | 3 | *-+--> | Nil |
   +---+---+    +---+---+    +---+---+    +-----+
    head=1                                  tail of last cell
```

That layout decides what is fast and what is slow.

| Operation | Cost | Notes |
|---|---|---|
| `head`, `tail`, `isEmpty` | constant | reads the first cell |
| Prepend (`x :: xs`) | constant | new cell pointing at the old list |
| Random access by index | linear | walks the chain |
| Reaching the last element | linear | walks the whole chain |
| Append (`xs :+ x`) | linear | walks the whole chain |

**Reach for `List` when:** you build front-to-back and consume front-to-back, especially in code that uses recursion or pattern matching. Pattern matching on `head :: tail` is the most idiomatic way to walk a `List` in Scala — notebook 06 leans on this.

In [ ]:
val xs = List(1, 2, 3)

// head / tail / isEmpty — constant time
println(xs.head)     // 1
println(xs.tail)     // List(2, 3)
println(xs.isEmpty)  // false

// Prepend with :: — constant time. Note the right-associative quirk:
//   `0 :: xs` is `xs.::(0)`, because methods ending in `:` bind to the right.
val ys = 0 :: xs
println(ys)          // List(0, 1, 2, 3)

// Build a list the long way — equivalent to List(1, 2, 3)
val explicit = 1 :: 2 :: 3 :: Nil
println(explicit)

// Append is linear — avoid in a tight loop
val zs = xs :+ 4
println(zs)          // List(1, 2, 3, 4)

## `Vector` — the default indexed sequence

`Vector` is internally a balanced 32-way tree. From the outside it behaves like an array. From the inside, the tree is so shallow that indexed access at any position is *effectively* constant time, even for very large vectors. Practically, you can treat `Vector` access as constant time and not worry about it.

| Operation | Cost | Notes |
|---|---|---|
| Random access by index | ~constant | log base 32, capped by tree depth |
| Append (`v :+ x`) | ~constant | |
| Prepend (`x +: v`) | ~constant | |
| `updated(i, x)` | ~constant | returns a new Vector with index `i` changed |

The operators have a tiny mnemonic. Where the colon sits is always *next to the collection*:

- `vec :+ x` — append (colon next to `vec`, so `vec` is the collection, `x` is the new element)
- `x +: vec` — prepend (colon next to `vec`, so `vec` is the collection, `x` is the new element)

**Reach for `Vector` when:** you need indexed access with a mix of prepend and append, or when you simply do not know exactly what your access pattern will be. **The rule of thumb: when in doubt between `List` and `Vector`, pick `Vector`.** Pick `List` only when you specifically want cons-cell ergonomics for recursion or pattern matching.

In [ ]:
val v = Vector(10, 20, 30, 40)

// Indexed access — effectively constant
println(v(2))               // 30

// Append, prepend, updated — all return a new Vector
println(v :+ 50)            // Vector(10, 20, 30, 40, 50)
println(0 +: v)             // Vector(0, 10, 20, 30, 40)
println(v.updated(1, 999))  // Vector(10, 999, 30, 40)

// The original is untouched
println(v)                  // Vector(10, 20, 30, 40)

## `Set` — uniqueness and membership

A `Set` holds unique elements with no defined iteration order. The fast operation is *membership testing* — asking "is this element in the set" is roughly constant time.

| Operation | Cost | Notes |
|---|---|---|
| `contains(x)` | ~constant | the headline operation |
| `+ x` | ~constant | new Set with `x` added |
| `- x` | ~constant | new Set with `x` removed |
| `union`, `intersect`, `diff` | linear in the smaller side | set-theory operators |

**The quietly powerful idea:** a `Set[A]` is also a function from `A` to `Boolean`. So if you have a `Set[String]` called `s`, you can call `s("red")` and it returns `true` or `false` — same as `s.contains("red")`. This means a `Set` can be passed directly anywhere a predicate is expected — to `filter`, to `exists`, anywhere. The higher-order machinery composes cleanly with sets.

In [ ]:
val colors = Set("red", "green", "blue")

// Membership
println(colors.contains("red"))   // true
println(colors("yellow"))         // false — Set as predicate

// Building new sets
println(colors + "yellow")        // Set(red, green, blue, yellow)
println(colors - "red")           // Set(green, blue)

// Set-theory operators
val warm = Set("red", "orange", "yellow")
println(colors.intersect(warm))   // Set(red)
println(colors.union(warm))       // Set(red, green, blue, orange, yellow)
println(colors.diff(warm))        // Set(green, blue)

// Set as predicate, applied directly to filter
val words = List("red car", "green light", "purple haze", "red rose")
val withColor = words.filter(w => colors.exists(c => w.contains(c)))
println(withColor)

## `Map` — key to value lookup

`Map` is the immutable hash map. It maps keys to values, with unique keys. You construct one with the arrow notation: `"alice" -> 30` produces a key-value pair where `"alice"` is the key and `30` is the value. Wrap a series of pairs in `Map(...)` and you have a map.

Lookup has two forms:

- **Unsafe form** — call the map like a function: `m("alice")`. Returns the value directly, or throws `NoSuchElementException` if the key is missing.
- **Safe form** — `m.get("alice")` returns an `Option[V]`. `Some(value)` if present, `None` if missing. Notebook 06 covers `Option` in depth.

There is also `m.getOrElse("alice", 0)` for "return the value if present, otherwise this default" without involving `Option`.

You modify a `Map` by deriving a new one. `m + (k -> v)` adds, `m - k` removes, `m.updated(k, v)` changes the value at an existing key. All return new maps.

In [ ]:
val ages = Map(
  "alice" -> 30,
  "bob"   -> 25,
  "carol" -> 42,
)

// Unsafe lookup — throws if absent
println(ages("alice"))           // 30

// Safe lookup — returns Option
println(ages.get("alice"))       // Some(30)
println(ages.get("dave"))        // None

// getOrElse — default value when absent
println(ages.getOrElse("dave", 0))  // 0

// Modifications return new maps
println(ages + ("dave" -> 18))   // adds dave
println(ages - "bob")            // removes bob
println(ages.updated("alice", 31))  // changes alice

// Iteration yields tuples
for (name, age) <- ages do
  println(s"$name is $age")

**About the arrow notation.** `"alice" -> 30` is not special syntax built into `Map`. It is a method available on every value — `->` is defined on `Any` and returns a 2-tuple. So `"alice" -> 30` returns the tuple `("alice", 30)`, and `Map(...)` is just a collection of such tuples. That is why iteration over a `Map` yields tuples, and why you can destructure them in a `for` expression with `for (name, age) <- ages`.

## `Array` — the mutable, fixed-size, java-flavoured one

`Array` is Scala's thin wrapper over a java array. It is mutable, fixed-size, and lives outside the immutable collections hierarchy. Use it when you need the exact memory layout that a contiguous block of primitives gives you, or when you are talking to a java application programming interface that expects an array.

Three things to register:

- **Indexing uses parentheses, not square brackets.** Square brackets in Scala always mean type parameters, never indices. So `arr(i)` gets you the element at position `i`.
- **Arrays are mutable.** A `val` binds the array, but the contents can still change in place. This is the gap notebook 02 hinted at — `val` locks the binding, not the value.
- **Arrays cannot grow.** They have a fixed length. If you need a resizable indexed sequence, use `Vector` for immutable or `mutable.ArrayBuffer` for mutable.

In practice you will see `Array` in three places: low-level performance code, java interop, and Spark internals — Spark stores partition data as primitive arrays for cache efficiency.

In [ ]:
val arr = Array(10, 20, 30)

// Indexing
println(arr(0))    // 10
println(arr(2))    // 30

// Mutation in place — even though it's a val
arr(0) = 99
println(arr(0))    // 99
// arr = Array(1, 2, 3)  // would not compile — val locks the binding

// Update functional-style (returns a NEW array, leaves original alone)
val updated = arr.updated(1, 999)
println(updated.mkString(", "))   // 99, 999, 30
println(arr.mkString(", "))       // 99, 20, 30 — original unchanged

// ArrayBuffer — mutable resizable
import scala.collection.mutable.ArrayBuffer
val buf = ArrayBuffer(1, 2, 3)
buf += 4
buf += 5
println(buf)       // ArrayBuffer(1, 2, 3, 4, 5)

## Tuples — small, fixed groupings

A tuple is not a collection in the same sense, but every `Map` entry is built on them, so it is worth meeting now. A tuple holds a small fixed number of values of *possibly different types*. You construct it with parentheses, and you access elements positionally with `_1`, `_2`, etc.

You can also destructure a tuple on the left side of a `val` — `val (name, age) = pair` binds both halves at once.

**When to switch away from tuples.** Tuples are great for two-or-three-value groupings used immediately. When you find yourself reaching for `_4`, or naming what each tuple field means in a comment, switch to a case class. Case classes give you named fields with the same convenience — notebook 05.

In [ ]:
// Tuple of mixed types
val person: (String, Int, Double) = ("ganesh", 30, 1.78)

// Positional access
println(person._1)       // ganesh
println(person._2)       // 30
println(person._3)       // 1.78

// Destructuring
val (name, age, height) = person
println(s"$name is $age, $height m")

// Swap helper for 2-tuples
val pair = ("a", 1)
println(pair.swap)       // (1, a)

## Picking a shape — the cheat sheet

| You want to ... | Reach for |
|---|---|
| Build front-to-back, walk recursively / pattern-match | `List` |
| Indexed access with mixed prepend and append | `Vector` (the default when in doubt) |
| Membership testing or deduplication | `Set` |
| Key-to-value lookup | `Map` |
| Java interop or contiguous primitives | `Array` |
| Mutable resizable buffer | `mutable.ArrayBuffer` |
| Two or three values bundled briefly | tuple |
| Named fields on a small record | case class (notebook 05) |

## Part 2 — Operations

The operations in this half apply uniformly across `List`, `Vector`, `Set`, `Map` values, `Array`, and even `String` (which behaves like an `IndexedSeq[Char]` for these purposes). Learn them once and they work everywhere.

The list of demo data below is reused across the next several cells.

In [ ]:
// A small dataset of bank transactions — reused below.
case class Tx(id: String, category: String, amount: Double, status: String)

val txs = List(
  Tx("T1", "grocery",  42.50,  "approved"),
  Tx("T2", "fuel",     58.00,  "approved"),
  Tx("T3", "grocery",  19.75,  "declined"),
  Tx("T4", "dining",   124.30, "approved"),
  Tx("T5", "fuel",     65.00,  "declined"),
  Tx("T6", "grocery",  31.20,  "approved"),
  Tx("T7", "dining",   89.00,  "approved"),
)

## `map`, `filter`, `flatMap` — the three you'll use every day

**`map`** — transform each element with a function. The shape is preserved.

**`filter`** — keep elements that satisfy a predicate. The shape is preserved.

**`flatMap`** — apply a function that returns a collection, then flatten one level. Useful when each input produces zero, one, or many outputs.

In [ ]:
// map — transform each element
val amounts: List[Double] = txs.map(_.amount)
println(amounts)

// filter — keep approved transactions
val approved: List[Tx] = txs.filter(_.status == "approved")
println(approved.map(_.id))

// chaining map and filter
val approvedAmounts = txs.filter(_.status == "approved").map(_.amount)
println(approvedAmounts)

// flatMap — each input produces a list of "items", then flattened
case class Order(id: String, lines: List[String])
val orders = List(
  Order("O1", List("apple", "bread")),
  Order("O2", List("coffee")),
  Order("O3", List("milk", "eggs", "butter")),
)
val allLines: List[String] = orders.flatMap(_.lines)
println(allLines)   // List(apple, bread, coffee, milk, eggs, butter)

## `fold` and `reduce` — aggregation

Both collapse a collection down to a single value by repeatedly combining elements. The difference is whether you provide an initial seed.

- **`fold(seed)(op)`** — start with `seed`, combine with each element. Works on empty collections (returns `seed`). The `foldLeft` variant fixes the traversal direction; `foldRight` goes the other way.
- **`reduce(op)`** — no seed; combine the first two elements, then the result with the third, etc. **Throws on an empty collection.** Use `reduceOption` to get back an `Option` instead.

When the seed type and the element type are the same, `reduce` is shorter to write. When they differ — say, accumulating a `Map` while iterating over a `List` — you need `fold`.

In [ ]:
val xs = List(1, 2, 3, 4, 5)

// reduce — sum
println(xs.reduce(_ + _))       // 15
println(xs.sum)                 // 15 — same thing, built-in shortcut
println(xs.product)             // 120 — same idea for multiplication

// fold — same sum, with a seed
println(xs.foldLeft(0)(_ + _))  // 15

// fold with a different accumulator type — build a string
val joined = xs.foldLeft("")((acc, x) => s"$acc-$x")
println(joined)                 // -1-2-3-4-5

// foldLeft and foldRight differ on non-commutative operations
println(xs.foldLeft(0)(_ - _))   // ((((0-1)-2)-3)-4)-5 = -15
println(xs.foldRight(0)(_ - _))  // 1-(2-(3-(4-(5-0)))) = 3

// reduceOption — safe on empty
println(List.empty[Int].reduceOption(_ + _))  // None
println(xs.reduceOption(_ + _))               // Some(15)

## `groupBy` and `groupMapReduce` — flat to keyed

**`groupBy(f)`** — partition the collection by the result of `f`. Returns a `Map[K, List[A]]` where the key is the result of `f` and the value is all the elements that produced that key.

**`groupMapReduce(key)(map)(reduce)`** — three-step shortcut: group by key, map each element, then reduce per group. Much cheaper than `groupBy` followed by `mapValues(_.map(...).reduce(...))` because it builds the result in one pass.

When you find yourself writing `.groupBy(...).view.mapValues(...).toMap`, switch to `groupMapReduce` — it expresses the intent more directly.

In [ ]:
// Group transactions by category
val byCategory: Map[String, List[Tx]] = txs.groupBy(_.category)
byCategory.foreach { (cat, ts) =>
  println(s"$cat -> ${ts.map(_.id).mkString(", ")}")
}

// Total amount per category — in one pass with groupMapReduce
val totalByCategory: Map[String, Double] =
  txs.groupMapReduce(_.category)(_.amount)(_ + _)
println(totalByCategory)

// Count per status
val countByStatus: Map[String, Int] =
  txs.groupMapReduce(_.status)(_ => 1)(_ + _)
println(countByStatus)

## `partition`, `span`, `splitAt` — slicing

**`partition(p)`** — splits into two collections — those where the predicate is true, and those where it is false. Returns a tuple of `(matching, notMatching)`.

**`span(p)`** — splits at the first element where the predicate becomes false. Everything *before* that point goes to the left tuple half; everything from there onward goes to the right.

**`splitAt(n)`** — splits at position `n`. First half is the first `n` elements, second half is everything else.

In [ ]:
val xs = List(1, 2, 3, 4, 5, 6, 7, 8, 9, 10)

// partition — interleaved
val (evens, odds) = xs.partition(_ % 2 == 0)
println(evens)  // List(2, 4, 6, 8, 10)
println(odds)   // List(1, 3, 5, 7, 9)

// span — stops at first false
val (small, rest) = xs.span(_ < 5)
println(small)  // List(1, 2, 3, 4)
println(rest)   // List(5, 6, 7, 8, 9, 10)

// splitAt — by position
val (first3, lastN) = xs.splitAt(3)
println(first3) // List(1, 2, 3)
println(lastN)  // List(4, 5, 6, 7, 8, 9, 10)

## `zip`, `unzip` — pairing two sequences

**`zip`** — pairs corresponding elements from two collections. Result has the length of the shorter one.

**`unzip`** — the reverse: given a collection of pairs, return two collections.

**`zipWithIndex`** — like `zip` against `(0, 1, 2, 3, ...)`, useful when you need positions.

In [ ]:
val names = List("alice", "bob", "carol")
val ages  = List(30, 25, 42, 99)   // longer — extra is dropped on zip

// zip — paired tuples
val paired: List[(String, Int)] = names.zip(ages)
println(paired)  // List((alice,30), (bob,25), (carol,42))

// unzip — back to two lists
val (n2, a2) = paired.unzip
println(n2)
println(a2)

// zipWithIndex — pair with position
val indexed = names.zipWithIndex
println(indexed) // List((alice,0), (bob,1), (carol,2))

## `sortBy`, `sortWith`, `sorted` — three flavours of ordering

**`sorted`** — sort using the natural `Ordering` of the element type. Works only when one is defined.

**`sortBy(f)`** — sort by a derived key. Cleaner than `sortWith` when one field decides everything.

**`sortWith(lt)`** — sort using a custom less-than function. Most flexible, slightly noisier.

In [ ]:
val nums = List(3, 1, 4, 1, 5, 9, 2, 6)
println(nums.sorted)              // List(1, 1, 2, 3, 4, 5, 6, 9)
println(nums.sorted.reverse)      // descending

// sortBy — by a derived key
val sortedByAmount = txs.sortBy(_.amount)
sortedByAmount.foreach(t => println(f"${t.id}  ${t.amount}%6.2f"))

// Descending sort — negate the numeric key, or use Ordering[X].reverse
val descAmount = txs.sortBy(-_.amount)
descAmount.foreach(t => println(f"${t.id}  ${t.amount}%6.2f"))

// sortWith — custom less-than
val byStatusThenAmount = txs.sortWith { (a, b) =>
  if a.status != b.status then a.status < b.status
  else a.amount < b.amount
}
byStatusThenAmount.foreach(t => println(f"${t.status}  ${t.id}  ${t.amount}%6.2f"))

## `take`, `drop`, `takeWhile`, `dropWhile`, `find`, `exists`, `forall`

A family of predicate-driven operations. They all stop as soon as the answer is decided — none of them walks the whole collection unless it has to.

- **`take(n)`** — first `n` elements. **`drop(n)`** — skip the first `n`.
- **`takeWhile(p)`** — elements from the start while `p` is true; stop at the first false.
- **`dropWhile(p)`** — skip from the start while `p` is true; keep from the first false onward.
- **`find(p)`** — first element satisfying `p`, returned as `Option`.
- **`exists(p)`** — true if any element satisfies `p`.
- **`forall(p)`** — true if every element satisfies `p`.

In [ ]:
val xs = List(1, 2, 3, 4, 5, 6, 7, 8, 9, 10)

println(xs.take(3))            // List(1, 2, 3)
println(xs.drop(3))            // List(4, 5, 6, 7, 8, 9, 10)
println(xs.takeWhile(_ < 5))   // List(1, 2, 3, 4)
println(xs.dropWhile(_ < 5))   // List(5, 6, 7, 8, 9, 10)

println(xs.find(_ > 7))        // Some(8)
println(xs.find(_ > 100))      // None

println(xs.exists(_ > 5))      // true
println(xs.forall(_ > 0))      // true
println(xs.forall(_ < 5))      // false

## `collect` — `map` plus `filter` in one pass

`collect` takes a *partial function* and applies it to every element. Elements where the partial function is not defined are dropped; elements where it is defined are transformed. Equivalent to `filter` then `map`, but in one pass and often clearer.

A partial function is written with `{ case ... => ... }` and only the patterns you list count as "defined." We will meet partial functions properly in notebook 06.

In [ ]:
val values: List[Any] = List(1, "two", 3.0, "four", 5, true, "six")

// Pull out only the strings, uppercased
val strings: List[String] = values.collect { case s: String => s.toUpperCase }
println(strings)   // List(TWO, FOUR, SIX)

// Equivalent two-step version (less efficient, less idiomatic)
val strings2 = values.filter(_.isInstanceOf[String]).map(_.toString.toUpperCase)
println(strings2)

// collect on the transactions — extract just the amounts of approved fuel
val approvedFuelAmounts: List[Double] = txs.collect {
  case Tx(_, "fuel", amount, "approved") => amount
}
println(approvedFuelAmounts)

## `for-yield` — and what it actually compiles to

`for-yield` is sugar. Anything you can write with `for-yield` can be written as a chain of `flatMap`, `map`, and `withFilter` calls — and the compiler does exactly that translation under the hood.

The translation rules:

- `for x <- xs yield expr` becomes `xs.map(x => expr)`
- `for x <- xs if p yield expr` becomes `xs.withFilter(x => p).map(x => expr)`
- Multiple generators become nested `flatMap` calls, with the *last* one becoming a `map`.

This is why `for-yield` works uniformly on any type that has `map`, `flatMap`, and `withFilter` — including `Option`, `Future`, `Try`, and your own types. We will use this in notebooks 06 and 09.

In [ ]:
// Single generator — same as map
val squares = for x <- List(1, 2, 3, 4) yield x * x
println(squares)   // List(1, 4, 9, 16)

// With a guard — same as filter then map
val evenSquares = for
  x <- List(1, 2, 3, 4, 5, 6)
  if x % 2 == 0
yield x * x
println(evenSquares)  // List(4, 16, 36)

// Two generators — nested
val pairs = for
  x <- List(1, 2, 3)
  y <- List("a", "b")
yield (x, y)
println(pairs)
// List((1,a), (1,b), (2,a), (2,b), (3,a), (3,b))

// The flatMap/map translation, explicit
val pairs2 = List(1, 2, 3).flatMap(x =>
  List("a", "b").map(y => (x, y))
)
println(pairs2)    // identical to `pairs`

## Lazy collections — `view`, `Iterator`, `LazyList`

Eager evaluation runs every step of a chain immediately. `xs.map(f).filter(p).take(3)` first allocates a full intermediate list from `map`, then a full filtered list, then takes three elements from that.

Three Scala types let you avoid those intermediate allocations.

- **`view`** — `xs.view.map(f).filter(p).take(3).toList` runs the chain lazily; nothing materializes until the final `.toList`. Use for chains over big collections when you only need part of the result.
- **`Iterator`** — single-pass, lazy by nature, no memoization. Reading from an `Iterator` consumes it.
- **`LazyList`** — a `List` where every element is lazily computed and memoized. Useful for representing potentially infinite sequences.

In [ ]:
val nums = (1 to 1_000_000).toList

// Eager — builds a million-element intermediate list, then takes 3
val eager = nums.map(_ * 2).filter(_ > 100).take(3)
println(eager)

// Lazy via view — short-circuits, never builds the intermediate
val lazyResult = nums.view.map(_ * 2).filter(_ > 100).take(3).toList
println(lazyResult)

// Iterator — single-pass
val it = Iterator(1, 2, 3, 4, 5)
println(it.map(_ * 10).toList)  // List(10, 20, 30, 40, 50)
// println(it.toList)            // Empty — the iterator is consumed

// LazyList — infinite sequence
val naturals: LazyList[Int] = LazyList.from(1)
println(naturals.take(5).toList)        // List(1, 2, 3, 4, 5)
println(naturals.map(_ * _).take(5).toList)  // List(1, 4, 9, 16, 25)

## Conversions and interop

Any Scala collection can be converted to any other compatible shape with `.to(Target)` — `.to(List)`, `.to(Vector)`, `.to(Set)`, `.to(Map)` (requires elements to be tuples). The older `.toList`, `.toVector`, `.toSet`, `.toMap` shortcuts still exist.

For java interop, import `scala.jdk.CollectionConverters.*` and use `.asScala` / `.asJava` at the boundary. Notebook 01 mentioned this; here it is in code.

In [ ]:
val ints = List(1, 2, 3, 2, 1)

println(ints.toSet)         // Set(1, 2, 3)
println(ints.to(Vector))    // Vector(1, 2, 3, 2, 1)
println(ints.to(Array).mkString(", "))

// List of pairs to Map
val pairs = List("a" -> 1, "b" -> 2, "c" -> 3)
println(pairs.to(Map))      // Map(a -> 1, b -> 2, c -> 3)

// Java interop
import scala.jdk.CollectionConverters.*
val javaList: java.util.List[Int] = ints.asJava
val backToScala: List[Int] = javaList.asScala.toList
println(backToScala)